In [9]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,association_rules

In [10]:
df = pd.read_csv('../processed/clientes_nuevos.csv', sep=';')
df_history = pd.read_csv('../raw/Antioquia.csv', sep=';')

In [11]:
df.info()
df_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970 entries, 0 to 969
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cliente              970 non-null    int64  
 1   recencia             970 non-null    int64  
 2   linea_favorita       970 non-null    object 
 3   marca_favorita       970 non-null    object 
 4   montoventapesos      970 non-null    float64
 5   margenpesos          970 non-null    float64
 6   porcentajedescuento  970 non-null    float64
 7   nombrecliente        970 non-null    object 
dtypes: float64(3), int64(2), object(3)
memory usage: 60.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 944178 entries, 0 to 944177
Data columns (total 31 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   factura              944178 non-null  object 
 1   cliente              944178 non-null  int64  
 2   nombrecliente

In [12]:
df = pd.merge(df[['cliente','recencia']], df_history, on='cliente', how='inner')
df_mba = df.groupby('factura')['referencia'].apply(list)
indices = df_mba.index.tolist()

In [14]:
te = TransactionEncoder()
te_ary = te.fit(df_mba).transform(df_mba)
df_mba_analysis = pd.DataFrame(te_ary, columns= te.columns_, index = indices)

In [17]:
df_apriori = apriori(df_mba_analysis,min_support=0.05, use_colnames=True)
df_apriori['cantidad_productos'] = df_apriori['itemsets'].apply(lambda x: len(x))

In [18]:
df_apriori

,support,itemsets,cantidad_productos
0,0.062284,(CPSCL),1
1,0.100346,(RKACI2GR12),1
2,0.079585,(RKLNSGX1),1
3,0.072664,(SBLEDA6012WM10CLBLK),1
4,0.065744,(SBLEDA607WM10CLBLK),1
5,0.069204,(SBLEDA609WM10CLBLK),1
6,0.069204,(SBLEDT918W),1
7,0.076125,(SBLEDT927W),1
8,0.051903,(SLBSLE),1
9,0.065744,(SLJETT),1


In [21]:
df_reglas = association_rules(df_apriori,metric = 'confidence', min_threshold=0.5 )

In [23]:
df_reglas.to_csv('../reglas/nuevos/reglas.csv',sep=';')